In [7]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import pymupdf 

In [8]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "pdf_vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


need to convert pdf into intacted md

In [9]:
pdf_path = "shimano-test.pdf"
doc = pymupdf.open(pdf_path)
md_chunks = []
image_dir = "images"
os.makedirs(image_dir, exist_ok=True)

for page_num, page in enumerate(doc):
    text = page.get_text()
    images = page.get_images(full=True)
    md = text
    for img_index, img in enumerate(images):
        xref = img[0]
        pix = pymupdf.Pixmap(doc, xref)
        img_path = f"{image_dir}/page{page_num}_img{img_index}.png"
        pix.save(img_path)
        md += f"\n\n![Image]({img_path})\n"
    md_chunks.append(md)

len(md_chunks)

169

In [25]:
md_chunks[100]

'101\nINSTALLATION OF HYDRAULIC DISC BRAKE SYSTEM\n Installing the brake caliper \nChecking the caliper fixing screw C length\nRear (same for both 140 mm and 160 mm)\n(A)\n(z)\nInsert the brake caliper mounting bolts C \ninto the frame mount area, and make \nsure that the lengths of the protruding \nsections of the bolts are 13 mm.\n(z)\t 13 mm\n(A)\t  Brake caliper mounting bolt C\nNOTICE\n•• When using a bolt length selector, make \nsure the tip of the brake caliper mounting \nbolt C is within the range A.\nBrake caliper mounting bolt C\nBolt length selector\nA\n•• Do not use a washer when checking the \nlength of brake caliper mounting bolt C.\n•• The length of the brake caliper mounting \nbolt C used varies depending on thickness \nof the frame.\x08\n \nUse brake caliper mounting bolt C that is \nsuitable for the thickness of the frame.\nFrame \nthickness\nBrake \ncaliper \nmounting \nbolt C \nlength\nFrame \nthickness\nBrake caliper \nmounting bolt \nC length\nY-part\n10 mm\n23 mm

In [12]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(" ".join(md_chunks))
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 37,964


In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents(md_chunks)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 244 chunks
First chunk:

page_content='(English)
DM-R8050-06
Dealer's Manual
ROAD
MTB
Trekking
City Touring/
Comfort Bike
URBAN SPORT
E-BIKE
R8050 series
ULTEGRA
SW-R9150
SW-R9160
SW-R610
ST-R8050
ST-R8060
ST-R8070
FD-R8050
RD-R8050
BR-R8070
SM-EW90-A
SM-EW90-B
EW-RS910
EW-WU111
EW-SD50
EW-SD50-I
EW-JC130
SM-EWC2
SM-JC40
SM-JC41
SM-BTR1
BT-DN110
BT-DN110-A
BM-DN100
SM-BA01
SM-BCR1
SM-BCR2
SM-BCC1
SM-RT800'


In [ ]:
chunks[100]

In [16]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 244 documents


In [17]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 244 vectors with 384 dimensions in the vector store


In [ ]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
print(result['documents'][0])      # Your single document text
print(result['embeddings'][0])     # The embedding vector for that document
print(result['metadatas'][0])